# ElbowVision — YOLOv8-pose + ConvNeXt 訓練（本番レベル）

**実行前に必ず: ランタイム → ランタイムのタイプを変更 → T4 GPU**

| セル | 内容 |
|------|------|
| 1 | 環境セットアップ |
| 2 | Google Drive マウント・データ展開 |
| 3 | dataset.yaml パス修正 |
| 4 | データ確認（枚数・サイズ・ラベル） |
| 5 | YOLOv8s-pose 訓練 |
| 6 | 定量評価（mAP・PCK・角度誤差） |
| 7 | Bland-Altman プロット |
| 8 | 推論可視化 |
| 9 | ConvNeXt ポジショニングズレ回帰 訓練 |
| 10 | ConvNeXt 評価（Bland-Altman） |
| 11 | トラブルシューティング |

## キーポイント定義（6点）
| index | 名称 | AP可視 | LAT可視 |
|-------|------|--------|---------|
| 0 | humerus_shaft      | ✓ | ✓ |
| 1 | lateral_epicondyle | ✓ | ✓ |
| 2 | medial_epicondyle  | ✓ | △（occluded） |
| 3 | forearm_shaft      | ✓ | ✓ |
| 4 | radial_head        | ✓ | △（occluded） |
| 5 | olecranon          | △ | ✓ |

In [ ]:
# ── セル 1: 環境セットアップ ──────────────────────────────────────
!pip install ultralytics timm -q
!pip install matplotlib pandas -q

import sys
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（CPU）"}')

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

In [ ]:
# ── セル 2: Google Drive マウント・データ展開 ─────────────────────
from google.colab import drive
import zipfile, os

drive.mount('/content/drive')

# Drive 上の zip パス（ファイルが存在するか確認）
# DRR生成コマンド例:
#   python elbow-train/elbow_synth.py \
#     --ct_dir data/raw_dicom/ct/ --out_dir data/yolo_dataset_v2/ \
#     --n_ap 240 --n_lat 720 --target_size 256 --domain_aug --laterality R
#   zip -r yolo_dataset_v2.zip data/yolo_dataset_v2/
ZIP_PATH  = '/content/drive/MyDrive/ElbowVision/yolo_dataset_v2.zip'
DATA_PATH = '/content/yolo_dataset_v2'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'ZIP が見つかりません: {ZIP_PATH}')

print('展開中...')
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/')

train_n = len(os.listdir(f'{DATA_PATH}/images/train'))
val_n   = len(os.listdir(f'{DATA_PATH}/images/val'))
print(f'\n展開完了: train={train_n}枚  val={val_n}枚  合計={train_n+val_n}枚')

In [ ]:
# ── セル 3: dataset.yaml パス修正 ────────────────────────────────
import yaml

YAML_PATH = f'{DATA_PATH}/dataset.yaml'
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DATA_PATH
with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f)

print('dataset.yaml:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

# kpt_shape: [6, 3] であることを確認
assert cfg.get('kpt_shape') == [6, 3], f"kpt_shape が [6,3] ではありません: {cfg.get('kpt_shape')}"
print('\n✅ kpt_shape: [6, 3] 確認OK')

In [ ]:
# ── セル 4: データ確認（画像サイズ・ラベル形式・AP/LAT 比率） ──────
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# 画像サイズ確認
imgs = sorted(Path(f'{DATA_PATH}/images/train').glob('*.png'))[:10]
sizes = [cv2.imread(str(p)).shape[:2] for p in imgs]
print('画像サイズ (H, W):', set(sizes))

# AP/LAT 比率確認
summary_csv = f'{DATA_PATH}/dataset_summary.csv'
if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    print('\nデータ分布:')
    print(df.groupby(['split', 'view_type']).size().reset_index(name='count'))
    print('\n回旋誤差（rotation_error_deg）統計:')
    print(df['rotation_error_deg'].describe())
    print('\n屈曲角（flexion_deg）統計:')
    print(df['flexion_deg'].describe())

# ラベル形式確認（1サンプル）
label = list(Path(f'{DATA_PATH}/labels/train').glob('*.txt'))[0]
print(f'\nラベルサンプル ({label.name}):')
txt = label.read_text().strip()
print(txt)
parts = txt.split()
n_kp = (len(parts) - 5) // 3
print(f'\nキーポイント数: {n_kp}  ← 6 であるべき')
print('フォーマット: class cx cy w h  kp0x kp0y v  kp1x kp1y v  kp2x kp2y v  kp3x kp3y v  kp4x kp4y v  kp5x kp5y v')
print('キーポイント: 0=上腕骨幹, 1=外側上顆, 2=内側上顆, 3=前腕骨幹, 4=橈骨頭, 5=肘頭')

In [ ]:
# ── セル 5: YOLOv8s-pose 訓練 ────────────────────────────────────
# yolov8s（small）: nanoより精度高い。T4 GPUで 20〜40分/100epoch。

RUN_DIR  = '/content/drive/MyDrive/ElbowVision/runs'
RUN_NAME = 'elbow_phantom_v1'

model = YOLO('yolov8s-pose.pt')

results = model.train(
    data     = YAML_PATH,
    epochs   = 150,
    imgsz    = 256,        # target_size=256 に合わせる
    rect     = True,       # 非正方形画像をパディングなしで処理（効率化）
    batch    = 16,         # T4 GPU で安定
    device   = 0,
    project  = RUN_DIR,
    name     = RUN_NAME,
    patience = 30,         # early stopping
    # ── augmentation（訓練データの多様性向上） ──
    hsv_h    = 0.0,        # X線は色情報なし → HSV変動は無効
    hsv_s    = 0.0,
    hsv_v    = 0.3,        # 輝度変動のみ有効
    fliplr   = 0.5,        # 左右フリップ（flip_idx: [0,2,1,3,4,5] で上顆左右入れ替え）
    flipud   = 0.0,        # 上下フリップは不要（解剖学的方向性あり）
    mosaic   = 0.0,        # モザイクは医療画像に不適切
    mixup    = 0.0,
    scale    = 0.2,        # ±20% スケール変動
    translate= 0.1,        # ±10% 平行移動
    degrees  = 5.0,        # ±5° 回転（軽微な傾き）
    save     = True,
    plots    = True,       # 訓練曲線・confusion matrix を保存
)

print('\n訓練完了')
print(f'最良モデル: {RUN_DIR}/{RUN_NAME}/weights/best.pt')

In [ ]:
# ── セル 6: 定量評価（mAP・PCK・角度誤差） ───────────────────────
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BEST_PT = f'{RUN_DIR}/{RUN_NAME}/weights/best.pt'
model_best = YOLO(BEST_PT)

# ── 6-1: YOLO 標準評価（mAP50, mAP50-95） ────────────────────────
val_results = model_best.val(data=YAML_PATH, split='val', rect=True)
print('=== YOLO 評価指標 ===')
print(f'  box  mAP50:    {val_results.results_dict["metrics/mAP50(B)"]:.3f}')
print(f'  box  mAP50-95: {val_results.results_dict["metrics/mAP50-95(B)"]:.3f}')
print(f'  pose mAP50:    {val_results.results_dict["metrics/mAP50(P)"]:.3f}')
print(f'  pose mAP50-95: {val_results.results_dict["metrics/mAP50-95(P)"]:.3f}')

# ── 6-2: PCK（Percentage of Correct Keypoints） ───────────────────
val_img_dir   = Path(f'{DATA_PATH}/images/val')
val_label_dir = Path(f'{DATA_PATH}/labels/val')
val_imgs = sorted(val_img_dir.glob('*.png'))

# 6キーポイント全て評価
KP_NAMES   = ['humerus_shaft', 'lateral_epicondyle', 'medial_epicondyle',
               'forearm_shaft', 'radial_head', 'olecranon']
THRESHOLDS = [5, 10, 20]  # pixels

errors_px    = {name: [] for name in KP_NAMES}
angle_errors = []  # 角度誤差（°）

def kp_to_angle(kpts_px):
    """
    6キーポイント → 肘関節角度（°）
    AP像: humerus_shaft - condyle_mid - forearm_shaft の角度
    LAT像: condyle_mid - 軸に基づく角度（condyle_midを使用して共通計算）
    """
    hs, le, me, fs = kpts_px[0], kpts_px[1], kpts_px[2], kpts_px[3]
    mid = ((le[0]+me[0])/2, (le[1]+me[1])/2)
    def ang(p1, p2):
        return math.degrees(math.atan2(p2[1]-p1[1], p2[0]-p1[0]))
    a1 = ang(hs, mid)
    a2 = ang(mid, fs)
    diff = abs(a1-a2) % 360
    return 360-diff if diff > 180 else diff

for img_path in val_imgs[:100]:
    lbl_path = val_label_dir / img_path.with_suffix('.txt').name
    if not lbl_path.exists():
        continue

    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # GT キーポイント読み込み（6点）
    line = lbl_path.read_text().strip()
    parts = line.split()
    if len(parts) < 5 + 6*3:
        continue  # 6キーポイント未満はスキップ
    gt_kps = [(float(parts[5+i*3])*w, float(parts[5+i*3+1])*h) for i in range(6)]

    # 予測
    res = model_best.predict(img, conf=0.1, verbose=False)
    if not res or res[0].keypoints is None or res[0].keypoints.xy is None:
        continue
    kpts = res[0].keypoints.xy[0].cpu().numpy()
    if len(kpts) < 6:
        continue
    pred_kps = [(float(kpts[i][0]), float(kpts[i][1])) for i in range(6)]

    # 各キーポイント誤差
    for i, name in enumerate(KP_NAMES):
        err = math.sqrt((gt_kps[i][0]-pred_kps[i][0])**2 + (gt_kps[i][1]-pred_kps[i][1])**2)
        errors_px[name].append(err)

    # 角度誤差（humerus-condyle-forearm の3点角度）
    gt_angle   = kp_to_angle(gt_kps)
    pred_angle = kp_to_angle(pred_kps)
    angle_errors.append(abs(gt_angle - pred_angle))

print('\n=== PCK（Percentage of Correct Keypoints） ===')
all_errors = [e for errs in errors_px.values() for e in errs]
print(f'  {"キーポイント":<25}', end='')
for thr in THRESHOLDS:
    print(f'  PCK@{thr:2d}px', end='')
print(f'  MeanErr(px)')

for name in KP_NAMES:
    errs = errors_px[name]
    if not errs:
        continue
    print(f'  {name:<25}', end='')
    for thr in THRESHOLDS:
        pck = sum(e <= thr for e in errs) / len(errs) * 100
        print(f'  {pck:7.1f}%', end='')
    print(f'  {np.mean(errs):8.1f}')

print(f'  {"全キーポイント平均":<25}', end='')
for thr in THRESHOLDS:
    pck = sum(e <= thr for e in all_errors) / len(all_errors) * 100
    print(f'  {pck:7.1f}%', end='')
print(f'  {np.mean(all_errors):8.1f}')

if angle_errors:
    print(f'\n=== 角度誤差（humerus-condyle-forearm） ===')
    print(f'  MAE:  {np.mean(angle_errors):.1f}°')
    print(f'  RMSE: {np.sqrt(np.mean(np.array(angle_errors)**2)):.1f}°')
    print(f'  ±5°以内:  {sum(e<=5  for e in angle_errors)/len(angle_errors)*100:.1f}%')
    print(f'  ±10°以内: {sum(e<=10 for e in angle_errors)/len(angle_errors)*100:.1f}%')

# ── 6-3: 合格基準判定 ────────────────────────────────────────────
pose_map  = val_results.results_dict['metrics/mAP50(P)']
all_pck10 = sum(e<=10 for e in all_errors)/len(all_errors)*100 if all_errors else 0
angle_mae = np.mean(angle_errors) if angle_errors else 999

print('\n=== 合格基準判定（プレ研究・ファントム） ===')
criteria = [
    ('pose mAP50',  pose_map,   0.70, '> 0.70'),
    ('PCK@10px',    all_pck10,  80.0, '> 80%'),
    ('角度MAE',     angle_mae,  10.0, '< 10°'),
]
all_pass = True
for name, val, thr, desc in criteria:
    ok = (val < thr) if name == '角度MAE' else (val >= thr)
    mark = '✅' if ok else '❌'
    print(f'  {mark} {name}: {val:.2f} （目標: {desc}）')
    if not ok:
        all_pass = False
print(f'\n→ 総合判定: {"合格 🎉" if all_pass else "要改善 — セル11 トラブルシューティング参照"}')

In [ ]:
# ── セル 7: Bland-Altman プロット（角度推定バイアス確認） ─────────
import matplotlib.pyplot as plt

if len(angle_errors) >= 10:
    gt_angles_list   = []
    pred_angles_list = []

    for img_path in val_imgs[:100]:
        lbl_path = val_label_dir / img_path.with_suffix('.txt').name
        if not lbl_path.exists():
            continue
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        parts = lbl_path.read_text().strip().split()
        if len(parts) < 5 + 6*3:
            continue
        gt_kps  = [(float(parts[5+i*3])*w, float(parts[5+i*3+1])*h) for i in range(6)]
        res = model_best.predict(img, conf=0.1, verbose=False)
        if not res or res[0].keypoints is None:
            continue
        kpts = res[0].keypoints.xy[0].cpu().numpy()
        if len(kpts) < 6:
            continue
        pred_kps = [(float(kpts[i][0]), float(kpts[i][1])) for i in range(6)]
        gt_angles_list.append(kp_to_angle(gt_kps))
        pred_angles_list.append(kp_to_angle(pred_kps))

    gt_arr   = np.array(gt_angles_list)
    pred_arr = np.array(pred_angles_list)
    diff     = pred_arr - gt_arr
    mean_val = (gt_arr + pred_arr) / 2
    bias     = diff.mean()
    loa      = 1.96 * diff.std()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.scatter(mean_val, diff, alpha=0.5, s=20)
    ax.axhline(bias,     color='red',    lw=2,   label=f'Bias: {bias:.1f}°')
    ax.axhline(bias+loa, color='orange', lw=1.5, ls='--', label=f'+1.96σ: {bias+loa:.1f}°')
    ax.axhline(bias-loa, color='orange', lw=1.5, ls='--', label=f'-1.96σ: {bias-loa:.1f}°')
    ax.axhline(0,        color='gray',   lw=0.5, ls=':')
    ax.set_xlabel('Mean angle (°)')
    ax.set_ylabel('Pred - GT (°)')
    ax.set_title('Bland-Altman Plot (YOLO キーポイント角度)')
    ax.legend(fontsize=9)

    ax = axes[1]
    lim = [min(gt_arr.min(), pred_arr.min())-5, max(gt_arr.max(), pred_arr.max())+5]
    ax.scatter(gt_arr, pred_arr, alpha=0.5, s=20)
    ax.plot(lim, lim, 'r--', label='Perfect')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('GT angle (°)')
    ax.set_ylabel('Predicted angle (°)')
    ax.set_title(f'GT vs Pred  (MAE={np.mean(np.abs(diff)):.1f}°, Bias={bias:.1f}°)')
    ax.legend()

    plt.tight_layout()
    out_fig = f'{RUN_DIR}/{RUN_NAME}/bland_altman_yolo.png'
    plt.savefig(out_fig, dpi=150)
    plt.show()
    print(f'保存: {out_fig}')
    print(f'Bland-Altman: Bias={bias:.1f}°, 95%LoA=[{bias-loa:.1f}°, {bias+loa:.1f}°]')
else:
    print('評価サンプルが不足しています（最低10枚必要）')

In [ ]:
# ── セル 8: 推論可視化（検証データ 6 枚） ────────────────────────
from IPython.display import Image as IPImage, display
import shutil, os

pred_out = '/content/preds'
os.makedirs(pred_out, exist_ok=True)

for p in sorted(val_imgs)[:6]:
    model_best.predict(str(p), conf=0.3,
                       save=True, project=pred_out, name='out', exist_ok=True)

for pred in sorted(Path(f'{pred_out}/out').glob('*.png')):
    print(pred.name)
    display(IPImage(str(pred)))

pred_drive = f'{RUN_DIR}/{RUN_NAME}/val_predictions'
shutil.copytree(f'{pred_out}/out', pred_drive, dirs_exist_ok=True)
print(f'\nDrive に保存: {pred_drive}')

## ConvNeXt ポジショニングズレ回帰モデル

YOLOv8でキーポイントを検出 → ConvNeXtで**直接**ポジショニングズレ量を回帰。

| 出力 | 説明 | 有効な像 |
|------|------|---------|
| rotation_error_deg | 理想位からの前腕回旋ズレ（°）| AP像 |
| flexion_deg | 肘屈曲角（°）| LAT像 |

**損失マスク**: AP像では flexion_deg を無視、LAT像では rotation_error_deg を無視。

In [ ]:
# ── セル 10: ConvNeXt ポジショニングズレ回帰 訓練 ────────────────
# ConvNeXt-Small で rotation_error_deg / flexion_deg を回帰
# ラベル: data/yolo_dataset_v2/convnext_labels.csv

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd, numpy as np, os
from pathlib import Path

# ─── モデル定義（API の convnext_model.py と同一定義） ───────────
OUTPUT_DIM   = 2
OUTPUT_NAMES = ["rotation_error_deg", "flexion_deg"]

class ElbowConvNeXt(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.convnext_small(weights=weights)
        in_features = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 128),
            nn.GELU(),
            nn.Linear(128, OUTPUT_DIM),
        )
    def forward(self, x):
        return self.backbone(x)

# ─── Dataset ─────────────────────────────────────────────────────
class ElbowDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, str(row['filename']))).convert('RGB')
        if self.transform:
            img = self.transform(img)
        view = str(row.get('view_type', 'AP')).upper()
        angles = torch.tensor([
            float(row.get('rotation_error_deg', 0.0)),
            float(row.get('flexion_deg', 90.0)),
        ], dtype=torch.float32)
        # AP: rotation_error_degのみ有効, LAT: flexion_degのみ有効
        mask = torch.tensor([
            1.0 if view == 'AP'  else 0.0,
            1.0 if view == 'LAT' else 0.0,
        ], dtype=torch.float32)
        return img, angles, mask

# ─── 設定 ────────────────────────────────────────────────────────
CONVNEXT_CSV  = f'{DATA_PATH}/convnext_labels.csv'
IMG_DIR_TRAIN = f'{DATA_PATH}/images/train'
IMG_DIR_VAL   = f'{DATA_PATH}/images/val'
CONVNEXT_OUT  = f'{RUN_DIR}/{RUN_NAME}/elbow_convnext_best.pth'
EPOCHS        = 80
BATCH         = 16
LR            = 1e-4
PATIENCE      = 20
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {device}')

# train/val 分割
df_all   = pd.read_csv(CONVNEXT_CSV)
df_train = df_all[df_all['split'] != 'val'].reset_index(drop=True)
df_val   = df_all[df_all['split'] == 'val'].reset_index(drop=True)
print(f'train: {len(df_train)}枚  val: {len(df_val)}枚')

# 分割 CSV を一時保存（img_dir で分けて処理）
df_train.to_csv('/tmp/_cx_train.csv', index=False)
df_val.to_csv(  '/tmp/_cx_val.csv',   index=False)

# img_dir: train/val 両方を包括するため DATA_PATH を使用
# filename カラムが "elbow_XXXXX.png" のみであれば images/ 配下を検索
# elbow_synth.py 出力では split カラムで分岐しているので img_dir はルート images/
def find_img_dir(data_path):
    """split 混在の CSV でも動くよう images/ 直下を検索するためのパス"""
    # split ごとに images/train・images/val に分かれているため
    # Dataset 内でファイルを見つけるには両方を試みる必要がある
    return data_path  # __getitem__ で split ディレクトリを考慮

class ElbowDatasetAuto(Dataset):
    """images/train/ と images/val/ の両方から filename を解決する Dataset"""
    def __init__(self, csv_file, data_path, transform=None):
        self.df        = pd.read_csv(csv_file)
        self.data_path = data_path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row['filename'])
        split = str(row.get('split', 'train'))
        img_path = os.path.join(self.data_path, 'images', split, fname)
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        view = str(row.get('view_type', 'AP')).upper()
        angles = torch.tensor([
            float(row.get('rotation_error_deg', 0.0)),
            float(row.get('flexion_deg', 90.0)),
        ], dtype=torch.float32)
        mask = torch.tensor([
            1.0 if view == 'AP'  else 0.0,
            1.0 if view == 'LAT' else 0.0,
        ], dtype=torch.float32)
        return img, angles, mask

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = ElbowDatasetAuto('/tmp/_cx_train.csv', DATA_PATH, train_tf)
val_ds   = ElbowDatasetAuto('/tmp/_cx_val.csv',   DATA_PATH, val_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

# ─── 訓練 ────────────────────────────────────────────────────────
cx_model  = ElbowConvNeXt(pretrained=True).to(device)
optimizer = optim.AdamW(cx_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val  = float('inf')
patience_cnt = 0
os.makedirs(os.path.dirname(CONVNEXT_OUT), exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    cx_model.train()
    t_loss = 0.0
    for imgs, targets, masks in train_dl:
        imgs, targets, masks = imgs.to(device), targets.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = cx_model(imgs)
        loss  = (torch.abs(preds - targets) * masks).mean()
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_dl)

    cx_model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for imgs, targets, masks in val_dl:
            imgs, targets, masks = imgs.to(device), targets.to(device), masks.to(device)
            v_loss += (torch.abs(cx_model(imgs) - targets) * masks).mean().item()
    v_loss /= max(len(val_dl), 1)
    scheduler.step()

    if epoch % 10 == 0 or epoch <= 5:
        print(f'Epoch {epoch:3d}/{EPOCHS}  train={t_loss:.2f}°  val={v_loss:.2f}°')

    if v_loss < best_val:
        best_val = v_loss
        patience_cnt = 0
        torch.save(cx_model.state_dict(), CONVNEXT_OUT)
        print(f'  → Best saved ({best_val:.2f}°)')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nConvNeXt 訓練完了。最良val loss: {best_val:.2f}°')
print(f'保存先: {CONVNEXT_OUT}')

In [ ]:
# ── セル 11: ConvNeXt 評価（AP 回旋ズレ / LAT 屈曲角 Bland-Altman） ─
import matplotlib.pyplot as plt

cx_model.load_state_dict(torch.load(CONVNEXT_OUT, map_location=device))
cx_model.eval()

gt_rot, pred_rot = [], []   # AP: rotation_error_deg
gt_flex, pred_flex = [], []  # LAT: flexion_deg

val_df = pd.read_csv('/tmp/_cx_val.csv')

with torch.no_grad():
    for _, row in val_df.iterrows():
        fname = str(row['filename'])
        split = str(row.get('split', 'val'))
        img_path = os.path.join(DATA_PATH, 'images', split, fname)
        if not os.path.exists(img_path):
            continue
        img = val_tf(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
        pred = cx_model(img)[0].cpu().numpy()
        view = str(row.get('view_type', 'AP')).upper()
        if view == 'AP':
            gt_rot.append(float(row['rotation_error_deg']))
            pred_rot.append(float(pred[0]))
        else:
            gt_flex.append(float(row['flexion_deg']))
            pred_flex.append(float(pred[1]))

def bland_altman_panel(ax1, ax2, gt, pred, title_prefix):
    gt_a, pr_a = np.array(gt), np.array(pred)
    diff   = pr_a - gt_a
    mean_v = (gt_a + pr_a) / 2
    bias   = diff.mean()
    loa    = 1.96 * diff.std()
    mae    = np.mean(np.abs(diff))

    ax1.scatter(mean_v, diff, alpha=0.5, s=20)
    ax1.axhline(bias,     color='red',    lw=2,   label=f'Bias={bias:.1f}°')
    ax1.axhline(bias+loa, color='orange', lw=1.5, ls='--', label=f'+1.96σ={bias+loa:.1f}°')
    ax1.axhline(bias-loa, color='orange', lw=1.5, ls='--', label=f'-1.96σ={bias-loa:.1f}°')
    ax1.axhline(0, color='gray', lw=0.5, ls=':')
    ax1.set_xlabel('Mean (°)'); ax1.set_ylabel('Pred-GT (°)')
    ax1.set_title(f'Bland-Altman: {title_prefix}'); ax1.legend(fontsize=8)

    lim = [min(gt_a.min(), pr_a.min())-5, max(gt_a.max(), pr_a.max())+5]
    ax2.scatter(gt_a, pr_a, alpha=0.5, s=20)
    ax2.plot(lim, lim, 'r--', label='Perfect')
    ax2.set_xlim(lim); ax2.set_ylim(lim)
    ax2.set_xlabel('GT (°)'); ax2.set_ylabel('Pred (°)')
    ax2.set_title(f'GT vs Pred: {title_prefix}  MAE={mae:.1f}°'); ax2.legend(fontsize=8)
    return mae, bias, loa

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

mae_rot = mae_flex = None
if gt_rot:
    mae_rot, bias_rot, loa_rot = bland_altman_panel(axes[0,0], axes[0,1], gt_rot, pred_rot, '回旋ズレ (AP)')
    print(f'AP  rotation_error_deg: n={len(gt_rot)}  MAE={mae_rot:.1f}°  Bias={bias_rot:.1f}°  LoA=[{bias_rot-loa_rot:.1f}, {bias_rot+loa_rot:.1f}]°')

if gt_flex:
    mae_flex, bias_flex, loa_flex = bland_altman_panel(axes[1,0], axes[1,1], gt_flex, pred_flex, '屈曲角 (LAT)')
    print(f'LAT flexion_deg:        n={len(gt_flex)}  MAE={mae_flex:.1f}°  Bias={bias_flex:.1f}°  LoA=[{bias_flex-loa_flex:.1f}, {bias_flex+loa_flex:.1f}]°')

plt.tight_layout()
out_fig = f'{RUN_DIR}/{RUN_NAME}/bland_altman_convnext.png'
plt.savefig(out_fig, dpi=150)
plt.show()
print(f'\n保存: {out_fig}')

# 合格基準
print('\n=== ConvNeXt 合格基準 ===')
for label, mae in [('rotation_error_deg MAE', mae_rot), ('flexion_deg MAE', mae_flex)]:
    if mae is not None:
        ok = mae < 10.0
        print(f'  {"✅" if ok else "❌"} {label}: {mae:.1f}° （目標 < 10°）')

## トラブルシューティング

### pose mAP50 < 0.70
- データ量を増やす: `--n_ap 240 --n_lat 720`（計1200枚）
- `--domain_aug` フラグが有効か確認
- epochs=200 に増やす

### PCK@10px < 80%（6キーポイント）
- radial_head / olecranon の自動検出が失敗している可能性
- `dataset_summary.csv` でキーポイント座標分布を確認
- `auto_detect_landmarks` のデバッグログを確認
- `create_phantom.py` の解剖学的パラメータを見直す

### Bland-Altman Bias > 5°
- DRR生成の透視投影ジオメトリとAPI角度計算が一致しているか確認
- `make_yolo_label` の `_project_kp_perspective` を検証

### ConvNeXt val loss が下がらない
- データ量不足（AP/LAT各200枚以上が目安）
- LR を下げる（1e-4 → 5e-5）
- `--domain_aug` でデータ多様性を増やす

### GPU メモリエラー
- YOLO: batch=8、imgsz=192 に下げる
- ConvNeXt: BATCH=8 に下げる

## 合格基準まとめ（プレ研究・ファントム）

| モデル | 指標 | 合格ライン |
|--------|------|----------|
| YOLOv8-pose | pose mAP50 | > 0.70 |
| YOLOv8-pose | PCK@10px（全6KP平均） | > 80% |
| YOLOv8-pose | 角度 MAE | < 10° |
| YOLOv8-pose | Bland-Altman Bias | < 5° |
| ConvNeXt | rotation_error_deg MAE | < 10° |
| ConvNeXt | flexion_deg MAE | < 10° |